# BruSafe · Notebook 05 — Module M5: absolute quantification, cargo copies per particle

*Section numbering follows the CAPSID framework document; this notebook implements Sections 6.1–6.7.*

**iGEM Tec-Chihuahua 2026 · CAPSID framework**

The upstream modules produce ratios or closed shells. This notebook produces the measurement that converts them into absolute, dose-relevant quantities:

$$\mathrm{CPP} \;=\; \frac{\text{dPCR copies of RNase-protected cargo, per mL}}{\text{particle count, per mL}}$$

## Why this quantity

| Without CPP | With CPP |
|---|---|
| M4 gives $P(3\times)/P(0\times)$, a ratio with no absolute scale | absolute packaging fraction |
| M3 gives closed shells, empty or full | **loaded** VLP concentration |
| yield in mg/L of coat protein | **doses per litre of culture** |

It is also the one place at which the dPCR panel and the assembly module measure the same physical
object, which makes it a cross-check and not two unrelated assays.

## Purpose

1. **Design the measurement** so that it is not dominated by an avoidable error (§2–§4).
2. **Propagate it** into loaded-VLP concentration and doses per litre (§5).
3. **Identify the scale-up bottleneck**, that is, which of the six factors in the chain moves cost
   per dose most, so that effort is directed where it matters (§6).

> **Note.** §2.2 should be read before instrument time is booked. NTA, the obvious choice for the
> denominator, is marginal at 27 nm, and the least expensive route is more precise than particle
> counting. The conclusion is quantified and not asserted.

---
## 1 · Setup

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

RNG = np.random.default_rng(20261013)
N_DRAW = 60000
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 9})

# --- particle constants -----------------------------------------------------
M_CP        = 13.7e3        # g/mol, MS2 coat protein monomer
N_MONO_VLP  = 180.0         # monomers per T=3 shell
M_PROT_VLP  = M_CP*N_MONO_VLP          # 2.47 MDa of protein per shell
CARGO_NT    = 1227.0        # cargo mRNA length, nt (set to the construct in use)
M_NT        = 330.0         # g/mol per nucleotide (Na+ form)
M_CARGO     = CARGO_NT*M_NT
NA          = 6.02214076e23

print(f"VLP protein mass : {M_PROT_VLP/1e6:.2f} MDa")
print(f"cargo mass       : {M_CARGO/1e3:.0f} kDa ({CARGO_NT:.0f} nt)")
print(f"loaded particle  : {(M_PROT_VLP+M_CARGO)/1e6:.2f} MDa "
      f"({100*M_CARGO/(M_PROT_VLP+M_CARGO):.0f}% RNA by mass at 1 copy/particle)")

# ---- Figure style (team palette) -------------------------------------------
# The colours are sampled from the team wordmark. Figures refer to the names
# below instead of hex codes, so the whole set can be restyled from this cell.
# SF Pro is used where installed; matplotlib substitutes another font otherwise.
CREAM, PALE, STEEL = "#F4F3EF", "#CCD8E2", "#89A1C2"
SKY,   BLUE, MID   = "#4CA3DF", "#3E77E1", "#2C5BA5"
NAVY,  INK         = "#173779", "#122047"
from matplotlib.colors import LinearSegmentedColormap
BRAND_CMAP = LinearSegmentedColormap.from_list("tecchihuahua",
                                               [NAVY, MID, BLUE, SKY])
plt.rcParams.update({
    "font.family": ["SF Pro Text", "SF Pro Display", "SF Pro", "Helvetica Neue"],
    "mathtext.fontset": "custom", "mathtext.default": "regular",
    "mathtext.rm": "SF Pro Text", "mathtext.it": "SF Pro Text:italic",
    "mathtext.bf": "SF Pro Text:bold",
    "figure.facecolor": CREAM, "axes.facecolor": CREAM, "savefig.facecolor": CREAM,
    "text.color": INK, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.edgecolor": INK, "xtick.color": INK, "ytick.color": INK,
    "grid.color": STEEL, "grid.alpha": 0.35,
    "legend.facecolor": CREAM, "legend.edgecolor": STEEL, "legend.framealpha": 0.9,
    "axes.prop_cycle": plt.cycler(color=[NAVY, BLUE, SKY, STEEL, MID, PALE]),
})


---
## 2 · The two measurements

### 2.1 · Numerator: dPCR of RNase-protected cargo

$$C_{meas} = C_{true}\,\varepsilon_{rel}\,\varepsilon_{ext} \;+\; B$$

Three effects can bias the measurement, and each needs its own control:

| Factor | Effect | Control |
|---|---|---|
| $\varepsilon_{rel}$ — capsid disruption | incomplete release undercounts | spike a known RNA into intact VLPs, disrupt, recover |
| $\varepsilon_{ext}$ — extraction recovery | undercounts | external RNA spike through the full protocol |
| $B$ — protection carryover | surviving free RNA **over**counts | naked cargo RNA + RNase, no capsid |

The carryover control is the one most often omitted and the most important. $B$ is the only term
that biases the result upward, and an uncorrected upward bias in the numerator inflates every
downstream dose estimate.

**dPCR counting precision is not the limiting factor.** With 26 000 partitions the counting CV is
below 1% across the usable occupancy range. The uncertainty comes from sample preparation and not
from partition counting, as §3 shows quantitatively. Quoting the partition CV as the error bar
would understate the uncertainty by more than an order of magnitude.

**EvaGreen constraint.** Probe multiplexing is not possible, so cargo and reference cannot share a
well by colour. They can be resolved by amplicon length (amplitude discrimination) or run in
separate wells, in which case the between-well variance must be carried explicitly. Ratios from
separate wells should not be treated as duplexed.

In [ ]:
def dpcr_cv(lam, n_part=26000):
    # CV of the estimated concentration from Poisson partitioning
    return np.sqrt((np.exp(lam) - 1.0)/(n_part*lam**2))

lam = np.logspace(-2, 1.0, 200)
plt.figure(figsize=(4.8, 3.2))
for n, c in zip((8500, 26000), [BLUE, SKY]):
    plt.loglog(lam, 100*dpcr_cv(lam, n), color=c, label=f"{n:,} partitions")
plt.axvspan(0.1, 4.0, color="grey", alpha=.15, lw=0)
plt.axhline(1.0, color=INK, ls=":", lw=1)
plt.xlabel(r"mean occupancy $\lambda$ (copies per partition)")
plt.ylabel("counting CV (%)"); plt.legend(fontsize=7)
plt.title("dPCR counting precision (grey = usable range)")
plt.tight_layout(); plt.show()

lam_opt = lam[np.argmin(dpcr_cv(lam))]
print(f"optimal occupancy      : lambda = {lam_opt:.2f}  -> CV = {100*dpcr_cv(lam_opt):.2f}%")
print(f"CV across lambda 0.1-4 : {100*dpcr_cv(0.1):.2f}% to {100*dpcr_cv(4.0):.2f}%")
print("Samples should be diluted into the grey band; the dPCR counting step then adds negligible error.")

### 2.2 · Denominator: counting particles

| Method | Works at 27 nm? | Precision | Note |
|---|---|---|---|
| **NTA** | **marginal** | poor | practical floor ~30–50 nm for low-index biologicals; MS2 VLPs sit at or below it |
| TRPS | marginal | moderate | smallest pores ~40 nm |
| Negative-stain TEM | yes | counting-limited | adsorption and field-selection bias need a bead standard |
| SEC-MALS | yes | excellent | absolute molar mass; also separates loaded from empty by mass |
| **SEC peak + protein assay** | yes | **good** | least expensive; precision derived below |

The last row is the unexpected result. Converting the protein mass of the purified capsid peak to
particles through $M_{prot,VLP}$ is more precise than counting particles, because counting error
scales as $1/\sqrt{n}$ and rarely more than a few hundred particles are counted.

$$N_{particles} = \frac{m_{protein}\,N_A}{M_{prot,VLP}}$$

> **Caution: A280 alone should not be used on a loaded VLP.** RNA absorbs strongly at 280 nm, so
> A280 overstates protein. A two-wavelength deconvolution looks like a remedy, but §2.3 shows that it
> is ill-conditioned in exactly this regime. An RNA-insensitive protein assay (BCA or Bradford)
> should supply the numerator of the equation above, and A260 the RNA.

In [ ]:
def particles_from_protein(mass_prot_mg_per_mL):
    # mg/mL of capsid-peak protein -> particles per mL
    return mass_prot_mg_per_mL*1e-3*NA/M_PROT_VLP

def tem_cv(n_counted, overdispersion=1.8):
    # field-to-field clustering inflates the Poisson CV; 1.8 is a typical factor
    return overdispersion/np.sqrt(n_counted)

print(f"{'method':<28} {'CV':>8}   basis")
print(f"{'TEM, 200 particles':<28} {100*tem_cv(200):7.1f}%   counting + field clustering")
print(f"{'TEM, 1000 particles':<28} {100*tem_cv(1000):7.1f}%   counting + field clustering")
print(f"{'SEC peak + BCA':<28} {100*np.sqrt(.05**2+.05**2+.03**2):7.1f}%   assay 5%, integration 5%, dilution 3%")
print("\nCounting 1000 particles by TEM (about an afternoon of work) is still less")
print("precise than a protein assay on the same material. TEM is better used as the")
print("orthogonal check on shape and malformed fraction, which no protein assay can")
print("provide, while the protein assay supplies the particle number.")

### 2.3 · Why the A260/A280 deconvolution fails here

Two components are resolved with two wavelengths:

$$\begin{pmatrix}A_{260}\\A_{280}\end{pmatrix} =
\begin{pmatrix}\varepsilon^{260}_{prot} & \varepsilon^{260}_{RNA}\\
\varepsilon^{280}_{prot} & \varepsilon^{280}_{RNA}\end{pmatrix}
\begin{pmatrix}c_{prot}\\c_{RNA}\end{pmatrix}$$

RNA absorbs about 25 times more per unit mass than protein at 260 nm, so in a loaded VLP the
protein estimate is a small difference between two large numbers. The condition number below shows
how strongly a 1% absorbance error is amplified in the protein mass.

In [ ]:
# mass extinction coefficients, (mg/mL)^-1 cm^-1
E = np.array([[0.57, 25.0],      # A260: protein, RNA
              [1.00, 12.5]])     # A280: protein, RNA
cond = np.linalg.cond(E)
print(f"condition number of the 2x2 system : {cond:.1f}")
print(f"-> a 1% absorbance error becomes up to {cond:.0f}% error in the protein mass\n")

for copies in (0, 1, 2, 4):
    m_rna = copies*M_CARGO/M_PROT_VLP          # RNA:protein mass ratio
    c = np.array([1.0, m_rna])
    A = E @ c
    print(f"  {copies} copies/particle: A260/A280 = {A[0]/A[1]:.3f}, "
          f"RNA is {100*E[0,1]*c[1]/A[0]:4.1f}% of A260 and {100*E[1,1]*c[1]/A[1]:4.1f}% of A280")
print("\nA260/A280 is a good readout for RNA per particle, because RNA dominates both")
print("channels, and a poor readout for protein. The job is therefore split: A260 for RNA,")
print("BCA for protein. A260 counts all packaged RNA, including host RNA, so comparing it")
print("with the dPCR cargo count gives the packaged-RNA specificity at no extra cost (§5).")
print("")

---
## 3 · The estimator and the sources of its error

$$\widehat{\mathrm{CPP}} = \frac{(C_{meas}-B)/(\varepsilon_{rel}\varepsilon_{ext})}{N_{particles}}$$

The error is propagated by Monte Carlo below. The four `MEASURED_` blocks should be replaced with
the results of the actual controls; the values given here are plausible placeholders and not data.

In [ ]:
# ---- SWAP POINT: replace each value with the measured result of the control ---
MEASURED_release   = (0.90, 0.06)     # mean, sd  (spike into intact VLP, disrupt, recover)
MEASURED_extract   = (0.72, 0.08)     # mean, sd  (external RNA spike, full protocol)
MEASURED_carry_pct = (0.03, 0.015)    # fraction of the signal that is unprotected carryover
MEASURED_prot_CV   = 0.077            # SEC peak + BCA, from §2.2

C_meas_per_mL = 2.5e13                # dPCR, protected cargo copies per mL
PROT_mg_per_mL = 0.25                 # capsid-peak protein

eps_rel  = np.clip(RNG.normal(*MEASURED_release, N_DRAW), 0.05, 1.0)
eps_ext  = np.clip(RNG.normal(*MEASURED_extract, N_DRAW), 0.05, 1.0)
carry    = np.clip(RNG.normal(*MEASURED_carry_pct, N_DRAW), 0.0, 0.9)
lam_used = 1.5
C_meas   = C_meas_per_mL*(1 + RNG.normal(0, dpcr_cv(lam_used), N_DRAW))
prot     = PROT_mg_per_mL*(1 + RNG.normal(0, MEASURED_prot_CV, N_DRAW))

C_true = C_meas*(1.0 - carry)/(eps_rel*eps_ext)
N_part = particles_from_protein(prot)
CPP    = C_true/N_part

def q(x): return np.percentile(x, [2.5, 50, 97.5])
lo, mdn, hi = q(CPP)
print(f"particles per mL        : {np.median(N_part):.3g}")
print(f"CARGO COPIES PER PARTICLE: {mdn:.3f}   95% [{lo:.3f}, {hi:.3f}]   "
      f"span {hi/lo:.2f}x")

logs = {"capsid disruption": np.log(eps_rel), "extraction recovery": np.log(eps_ext),
        "protection carryover": np.log(1-carry), "dPCR counting": np.log(C_meas/C_meas_per_mL),
        "particle count (protein)": np.log(prot/PROT_mg_per_mL)}
tot = np.var(np.log(CPP))
print("\nERROR BUDGET (share of log-variance)")
for k, v in sorted(logs.items(), key=lambda kv: -np.var(kv[1])):
    print(f"  {k:<26s} {100*np.var(v)/tot:5.1f} %")
print("\ndPCR counting is negligible. The recovery controls dominate, which is")
print("why §2.1 insists on running them rather than assuming efficiency = 1.")

---
## 4 · Design: what precision costs

CPP multiplies directly into doses per litre, so its CV adds in quadrature to all the other
contributions in §6. The scan below shows what tightening each control buys.

In [ ]:
def cpp_cv(rel_sd=0.06, ext_sd=0.08, carry_sd=0.015, prot_cv=0.077,
           n_tech=1, n_bio=3, n=40000, rng=RNG):
    # technical replicates average the dPCR + protein steps; biological
    # replicates average everything, including the recovery controls
    r  = np.clip(rng.normal(0.90, rel_sd/np.sqrt(n_bio), n), .05, 1)
    e  = np.clip(rng.normal(0.72, ext_sd/np.sqrt(n_bio), n), .05, 1)
    cy = np.clip(rng.normal(0.03, carry_sd/np.sqrt(n_bio), n), 0, .9)
    c  = 1 + rng.normal(0, dpcr_cv(1.5)/np.sqrt(n_tech*n_bio), n)
    p  = 1 + rng.normal(0, prot_cv/np.sqrt(n_tech*n_bio), n)
    x  = c*(1-cy)/(r*e*p)
    return float(np.std(np.log(x)))

rows = [dict(scenario="no controls (assume eps = 1)", cv=cpp_cv(rel_sd=.20, ext_sd=.25, carry_sd=.10)),
        dict(scenario="controls, 3 bio reps",         cv=cpp_cv()),
        dict(scenario="controls, 6 bio reps",         cv=cpp_cv(n_bio=6)),
        dict(scenario="controls + duplicate wells",   cv=cpp_cv(n_bio=6, n_tech=2))]
d = pd.DataFrame(rows); d["CV_%"] = 100*d.cv; d["95%_span"] = np.exp(2*1.96*d.cv)
display(d[["scenario", "CV_%", "95%_span"]].round(2))
print("Running the recovery controls is worth more than any amount of replication")
print("without them. Beyond ~6 biological replicates the return flattens, and further")
print("effort is better spent on the bottleneck identified in §6.")

---
## 5 · From CPP to loaded VLPs, and packaging specificity

Two quantities follow immediately:

$$f_{loaded} = \min(\mathrm{CPP},\,1)\ \text{(if } \le 1 \text{ copy fits)}, \qquad
\text{specificity} = \frac{\mathrm{CPP}\times M_{cargo}}{\text{total RNA per particle from } A_{260}}$$

The second comes at no extra cost. A260 counts **all** packaged RNA and dPCR counts only the cargo,
so their ratio is the fraction of packaged RNA that is the construct. This is the absolute
packaging fraction that M4 could give only as a ratio.

In [ ]:
A260_per_particle_ng = 0.0            # set from the A260 of the capsid peak; 0 = skip
TOTAL_RNA_nt_per_particle = 1600.0    # from A260/BCA on the SEC peak; placeholder

f_loaded = np.clip(CPP, 0, 1)
specificity = np.clip(CPP*CARGO_NT/TOTAL_RNA_nt_per_particle, 0, 1)
lo, mdn, hi = q(f_loaded); print(f"loaded fraction        : {mdn:.3f}  95% [{lo:.3f}, {hi:.3f}]")
lo, mdn, hi = q(specificity); print(f"packaging specificity  : {mdn:.3f}  95% [{lo:.3f}, {hi:.3f}]")
print("\nSpecificity is the number M4's dPCR ladder cannot produce on its own -")
print("it needs the particle count. Cross-check it against the P(3x)/P(0x)")
print("enrichment: the two must be consistent, and if they are not, one of the")
print("two assays has a systematic error that has not yet been identified.")

---
## 6 · Scalability

The chain from culture volume to doses is

$$\text{doses/L} = \frac{[\mathrm{CP}]_{mg/L}\times f_{sol}\times f_{cap}\times f_{loaded}\times \eta_{purif}}{d_{mg}}$$

**Every input below is a placeholder.** The dose size in particular must come from the immunology
and human-practices work and not from this notebook. The purpose of the analysis is the tornado in
§6.2, which ranks the factors by their effect on cost per dose. This ranking is considerably more
robust than any absolute figure.

In [ ]:
# ---- SWAP POINT: process and product assumptions ---------------------------
CP_mg_per_L   = (178.0, 18.0)      # total CP at harvest, mean/sd (from M1)
F_SOL         = (0.438, 0.020)     # M2
# M3 at the measured contact energy. The spread is taken from the contact-schedule
# scan (0.31-0.95 at 18 h), because that scan is the uncertainty.
F_CAP         = (0.76, 0.16)       # M3 at dG = -3.82 kcal/mol, schedule-scan spread
F_TEM_OK      = (0.95, 0.04)       # correctly formed fraction, TEM, 27 +/- 2 nm
ETA_PURIF     = (0.45, 0.12)       # clarification + PEG + SEC, overall recovery
M_VLP_MDA     = 2.47               # MDa of protein per particle (180 x 13.7 kDa)
DOSE_MG       = (0.10, 0.03)       # mg VLP protein per dose (input from the immunology work)
COST_PER_L    = 4.0                # USD per L of culture, media + induction + labour

def draw(ms, n=N_DRAW, lo=1e-6, hi=None):
    x = RNG.normal(ms[0], ms[1], n)
    return np.clip(x, lo, hi if hi is not None else np.inf)

cp   = draw(CP_mg_per_L)
fs   = np.clip(draw(F_SOL), 0, 1)
fc   = np.clip(draw(F_CAP), 0, 1)
fl   = np.clip(RNG.choice(f_loaded, N_DRAW), 0, 1)
eta  = np.clip(draw(ETA_PURIF), 0, 1)
ftem = np.clip(draw(F_TEM_OK), 0, 1)
dose = draw(DOSE_MG)

# PRIMARY OUTPUT: concentration of correctly formed, loaded particles at harvest
mg_loaded_per_L = cp*fs*fc*ftem*fl*eta
particles_per_mL = mg_loaded_per_L*1e-3/(M_VLP_MDA*1e6*1.6605e-24)/1e3
doses_per_L     = mg_loaded_per_L/dose
L_per_1000      = 1000.0/np.maximum(doses_per_L, 1e-9)
cost_per_dose   = COST_PER_L/np.maximum(doses_per_L, 1e-9)

print("VLP CONCENTRATION AT HARVEST  (the quantity the process is run for)")
for lab, x, u in [("loaded VLP protein", mg_loaded_per_L, "mg/L culture"),
                  ("loaded particles",   particles_per_mL, "particles/mL")]:
    lo_, md_, hi_ = q(x)
    print(f"  {lab:<24s} {md_:10.4g}   95% [{lo_:.3g}, {hi_:.3g}]  {u}")

print("\nDERIVED, not modelled: the figures below divide the line above by an assumed")
print("dose size that no experiment of ours constrains. Report them as planning")
print("numbers, never as a prediction of potency.")
for lab, x, u in [("doses per litre",    doses_per_L,     "doses/L"),
                  ("culture per 1000 doses", L_per_1000,  "L"),
                  ("culture cost per dose",  cost_per_dose, "USD (media+induction only)")]:
    lo_, md_, hi_ = q(x)
    print(f"  {lab:<24s} {md_:10.4g}   95% [{lo_:.3g}, {hi_:.3g}]  {u}")
print("\nCulture cost only. Purification consumables, QC, fill-finish and")
print("regulatory burden are not here and will dominate a real cost of goods.")

In [ ]:
# --- 6.2 tornado: which factor actually moves doses per litre? --------------
# Decomposed on the PRIMARY output -- VLP concentration at harvest -- so the
# answer does not depend on an assumed dose size. The dose term is reported
# separately because it only affects the derived planning figures.
base = {"total CP": cp, "soluble fraction": fs, "capsid fraction": fc,
        "correctly formed (TEM)": ftem, "loaded fraction": fl,
        "purification recovery": eta}
tot = np.var(np.log(mg_loaded_per_L))
share = {}
for k, v in base.items():
    s = np.var(np.log(np.maximum(v, 1e-12)))/tot
    share[k] = 100*s
ser = pd.Series(share).sort_values()

fig, ax = plt.subplots(figsize=(5.4, 3.0))
ax.barh(ser.index, ser.values, color=[NAVY if v > 20 else SKY for v in ser.values])
ax.set_xlabel("share of variance in log[VLP] at harvest (%)")
ax.set_title("What moves the harvest titre")
for i, v in enumerate(ser.values): ax.text(v+1, i, f"{v:.0f}%", va="center", fontsize=8)
fig.tight_layout(); plt.show()

print("The tornado is a guide to allocating effort and not a result. The two dark bars mark")
print("the factors for which additional work changes the answer; the pale ones are already")
print("good enough that improving them would change nothing measurable.")

_d = np.var(np.log(np.maximum(dose, 1e-12)))/np.var(np.log(doses_per_L))
print(f"\nDose size contributes {100*_d:.0f}% of the variance in doses per litre and, by")
print("construction, none of the variance in the harvest titre above.")

---
## 7 · Next steps

**Measurement sequence.**

1. Purify to a clean SEC capsid peak. Everything here assumes that the peak is free of dimer and
   partial shells, so the trace should be checked first.
2. Run the **three recovery controls** (§2.1). Omitting them costs more precision than any
   replication recovers (§4).
3. Split the readout: **BCA for protein, A260 for RNA, dPCR for cargo.** A280 should not be used on
   a loaded particle (§2.3), and NTA is not suitable as the particle counter at 27 nm (§2.2).
4. Use at least three biological replicates, and six if the schedule allows.
5. Use TEM as an orthogonal check on *shape and malformed fraction*, which no bulk assay provides,
   and not as the particle counter.

**Cross-checks that must agree, and the interpretation of a disagreement.**

- Specificity from §5 against the $P(3\times)/P(0\times)$ enrichment from Module M4. An
  inconsistency implies a systematic error in one of the two assays.
- Particles from protein mass against particles from TEM counts. A ratio far from 1 indicates
  adsorption bias in the TEM or non-capsid protein in the SEC peak.
- CPP well above 1 copy per particle implies either that the cargo is fragmenting and being counted
  more than once, or that the protection step is leaking. Both are numerator problems.

**Scope.** Every number in §5 and §6 is a placeholder. The outputs that do not depend on the
placeholders are the error budget in §3 and the ranking in §6.2; these are structural and can be
quoted now. The absolute figures cannot.
---
## 8 · References

1. Huggett JF, Foy CA, Benes V, et al. The digital MIQE guidelines: minimum information for publication of quantitative digital PCR experiments. *Clin Chem.* 2013;59(6):892–902.
2. Whale AS, Huggett JF, Tzonev S. Fundamentals of multiplexing with digital PCR. *Biomol Detect Quantif.* 2016;10:15–23.
3. Pinheiro LB, Coleman VA, Hindson CM, et al. Evaluation of a droplet digital PCR format for DNA copy number quantification. *Anal Chem.* 2012;84(2):1003–1011.
4. Filipe V, Hawe A, Jiskoot W. Critical evaluation of nanoparticle tracking analysis by NanoSight for the measurement of nanoparticles and protein aggregates. *Pharm Res.* 2010;27(5):796–810.
5. Vogt S, Stadlmayr G, Grillari J, Rüker F, Wozniak-Knopp G. Comparison of nanoparticle tracking and multi-angle DLS for VLP characterisation. *Methods Mol Biol.* 2016.
6. Some D, Amartely H, Tsadok A, Lebendiker M. Characterization of proteins by SEC-MALS. *J Vis Exp.* 2019;(148):e59615.
7. Porterfield JZ, Zlotnick A. A simple and general method for determining the protein and nucleic acid content of viruses by UV absorbance. *Virology.* 2010;407(2):281–288.
8. Pickett GG, Peabody DS. Encapsidation of heterologous RNAs by bacteriophage MS2 coat protein. *Nucleic Acids Res.* 1993;21(19):4621–4626.
9. Legendre D, Fastrez J. Production in *Saccharomyces cerevisiae* of MS2 virus-like particles packaging functional heterologous mRNAs. *J Biotechnol.* 2005;117(2):183–194.
10. Zeltins A. Construction and characterization of virus-like particles: a review. *Mol Biotechnol.* 2013;53(1):92–107.